# 08 · Los límites del paper
*A quién se alinea, las fallas que el paper muestra y lo que dejó abierto, con sus propias palabras*

Notebook del episodio 8 de **InstructGPT** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/instructgpt/limites-del-paper)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** 5.2: cuatro capas de «humanos» → acuerdo 73 % → q² + (1−q)² → techo ≈ 0,84 vs RM 72,4 / 69,6 → una etiqueta por par esconde los pares polémicos → win rate = Σ π_g w_g esconde a un grupo (0,63 vs 0,15) → figura 9 → RM Bradley-Terry sobre rasgos aprende a premiar cubrirse (P = 0,77 en una obvia) → «follow the user's instruction»: best-of-4 con RM útil vs cauto → entropía de pares (Winogender/CrowS): «respetuoso» baja 0,413 → 0,243 → 5.4 preguntas abiertas

**Cómo correrlo:** con el entorno de `~/Movies/serie-instructgpt/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "instructgpt"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*InstructGPT. Episodio ocho: los límites del paper. Un buen paper dice qué no resolvió, y este dedica una sección entera. Hoy la leemos con citas textuales: a quién se alinea el modelo, qué fallas muestran ellos mismos, y qué quedó abierto. Para cada límite, un experimento chico que muestra el mecanismo.*

## Parte 1 · A quién se alinea

La literatura habla de «valores humanos». El paper es más preciso: dice de quién son las preferencias.

### Cuatro capas de «humanos»

La sección cinco punto dos enumera cuatro capas. Primero, los etiquetadores: unas cuarenta personas, en su mayoría de habla inglesa, en Estados Unidos o el Sudeste asiático, contratadas por Upwork o Scale AI. Segundo, los investigadores: escriben las instrucciones de etiquetado y contestan dudas en un chat compartido. Tercero, los clientes de la API, application programming interface: sus prompts definen qué se entrena. Y cuarto, lo que falta: los usuarios finales y la gente afectada. Al principio, la API se usaba por lista de espera, sembrada con empleados de OpenAI.

| quién | cómo influye | lo que dice el paper |
|---|---|---|
| etiquetadores | escriben demos y eligen | ~40; EE. UU. y Sudeste asiático |
| investigadores | escriben las instrucciones | responden dudas en un chat |
| clientes de la API | eligen qué se pregunta | lo que ellos creen valioso |
| usuarios y afectados | no participan | la API tenía lista de espera |

> **Ouyang et al., 2022 — sección 5.2** · [arXiv:2203.02155](https://arxiv.org/abs/2203.02155)
>
> **«a set of labelers' preferences»** · «we have aligned to a set of labelers’ preferences that were influenced, among others things, by the instructions they were given, the context in which they received them (as a paid job), and who they received them from.»

La frase central. Se alineó a las preferencias de un grupo de etiquetadores, influidas por las instrucciones que recibieron, por el contexto, un trabajo pago, y por quién se las dio. No a «la humanidad».

### Cuánto pueden coincidir dos personas

El paper midió que dos etiquetadores coinciden alrededor del setenta y tres por ciento. ¿Qué techo le pone eso a un reward model? Simulamos. Cada par tiene una preferencia de fondo, y cada persona la sigue con probabilidad q, cero coma ochenta y cuatro. Una persona: acierta el gusto común o lo da vuelta. Dos personas juzgan los mismos pares. Y un reward model perfecto, que predice el gusto común. Las dos personas coinciden cero coma siete tres cuatro, como en el paper. Pero el modelo perfecto coincide con cada una cero coma ocho cuatro tres: más que entre ellas.

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep08_acuerdo.py
__file__ = 'snippets/ep08_acuerdo.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import numpy as np
rng = np.random.default_rng(0)

In [3]:
n, q = 20000, 0.84                        # q: cada persona coincide con el gusto común
comun = rng.random(n) < 0.5               # la preferencia de fondo en cada par
def etiquetar():                          # una persona: acierta el gusto común con prob. q
    return np.where(rng.random(n) < q, comun, ~comun)

a, b = etiquetar(), etiquetar()           # dos personas, los mismos pares
print(f"acuerdo entre dos personas:   {(a == b).mean():.3f}")
print(f"RM perfecto contra la persona a: {(comun == a).mean():.3f}")
print(f"RM que imita a la persona b:  {(b == a).mean():.3f}")

acuerdo entre dos personas:   0.734
RM perfecto contra la persona a: 0.843
RM que imita a la persona b:  0.734


### Del acuerdo al techo

$$
P(\text{acuerdo}) = q^2 + (1-q)^2
$$
<small>$q$: prob. de que una persona siga el gusto común</small>

$$
2q^2 - 2q + (1-a) = 0 \;\Rightarrow\; q = \tfrac12\big(1 + \sqrt{2a - 1}\big)
$$
<small>$a$: el acuerdo medido · fórmula de la cuadrática</small>

$$
\text{RM: } 72{,}4\,\% \;\text{y}\; 69{,}6\,\% \text{ con otras personas}
$$
<small>sección 4.1 del paper</small>

La cuenta. q es la probabilidad de que una persona siga la preferencia de fondo. Dos personas coinciden si las dos la siguen, q al cuadrado, o si las dos fallan, uno menos q al cuadrado. En el dibujo, las dos partes suman cero coma siete tres. Si a es el acuerdo medido, desarrollamos el cuadrado y pasamos todo a un lado: queda una cuadrática en q. La resolvemos con la fórmula de siempre y nos quedamos con la raíz mayor que un medio. Con a igual a cero coma siete tres, q da cero coma ocho cuatro. Los reward models del paper aciertan setenta y dos coma cuatro por ciento, y sesenta y nueve coma seis con personas que no aportaron datos. Debajo de ese techo, que es nuestro, no del paper.

### Una etiqueta por par esconde el desacuerdo

Otro límite que el paper reconoce: por costo, la mayoría de las comparaciones las etiquetó una sola persona. Pares de dos tipos: claros, donde el noventa y cinco por ciento elige A, y polémicos, sesenta contra cuarenta. Con una etiqueta, cada par tiene un ganador y nada más. Con tres personas por par. Un par se ve dividido si el voto sale dos a uno. Con tres etiquetas, el cuarenta y tres por ciento se ve dividido: catorce por ciento de los claros, setenta y uno de los polémicos. Con una, los polémicos son invisibles: el reward model los aprende como si hubiera consenso.

In [4]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep08_una_etiqueta.py
__file__ = 'snippets/ep08_una_etiqueta.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import numpy as np
rng = np.random.default_rng(0)

In [5]:
n = 6000
claro = rng.random(n) < 0.5               # la mitad de los pares: casi todos piensan igual
p_A = np.where(claro, 0.95, 0.60)         # prob. de que una persona elija A

una = rng.random(n) < p_A                 # el paper: una etiqueta por par
tres = (rng.random((n, 3)) < p_A[:, None]).sum(1)   # tres personas
dividido = (tres == 1) | (tres == 2)

print("con una etiqueta, desacuerdo visible: ninguno")
print(f"con tres: divididos {dividido.mean():.0%} de los pares")
print(f"  entre los claros {dividido[claro].mean():.0%},"
      f" entre los polémicos {dividido[~claro].mean():.0%}")

con una etiqueta, desacuerdo visible: ninguno
con tres: divididos 43% de los pares
  entre los claros 14%, entre los polémicos 71%


### El promedio esconde a un grupo

El paper advierte que alinear al promedio puede no ser deseable. Dos grupos de etiquetadores: ochenta y veinte por ciento. El grupo A prefiere el estilo X en tres de cada cuatro pares; el grupo B prefiere el estilo Y, casi siempre. Juntamos las etiquetas, como hace un dataset único. Y miramos el win rate de un modelo que siempre responde X, por grupo. Juntos, X gana cero coma sesenta y tres: parece un éxito. Para el grupo A, cero coma setenta y cinco. Para el grupo B, cero coma uno cinco: lo empeoramos casi siempre, y el número agregado no lo muestra.

In [6]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep08_grupos.py
__file__ = 'snippets/ep08_grupos.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import numpy as np
rng = np.random.default_rng(0)

In [7]:
peso = {"grupo A": 0.8, "grupo B": 0.2}   # cuánta gente de cada grupo etiqueta
prefiere_X = {"grupo A": 0.75, "grupo B": 0.15}   # estilo X contra estilo Y

n = 10000
grupo = np.where(rng.random(n) < peso["grupo A"], "grupo A", "grupo B")
p = np.vectorize(prefiere_X.get)(grupo)
elige_X = rng.random(n) < p
print(f"todas las etiquetas juntas: X gana {elige_X.mean():.2f}")
for g in peso:                            # el modelo que siempre responde X
    print(f"  win rate de X para el {g}: {elige_X[grupo == g].mean():.2f}")

todas las etiquetas juntas: X gana 0.63
  win rate de X para el grupo A: 0.75
  win rate de X para el grupo B: 0.15


### Un win rate es un promedio pesado

$$
\bar w = \sum_g \pi_g\, w_g = 0{,}8 \cdot 0{,}75 + 0{,}2 \cdot 0{,}15 = 0{,}63
$$
<small>$\pi_g$: fracción del grupo $g$ · $w_g$: su win rate</small>

$$
\min_g w_g = 0{,}15
$$
<small>lo que el promedio no muestra</small>

En fórmula. Pi sub g es qué fracción de las etiquetas viene del grupo g, y w sub g es el win rate dentro de ese grupo. El win rate total es el promedio pesado: cero coma ocho por cero coma setenta y cinco, más cero coma dos por cero coma quince. En el dibujo, el grupo grande domina la suma. Mirar el mínimo entre grupos, en vez del promedio, cuenta otra historia. El paper lo dice así: si un texto afecta sobre todo a una minoría, quizás convenga pesar más las preferencias de esa minoría.

> **Ouyang et al., 2022 — sección 5.2** · [arXiv:2203.02155](https://arxiv.org/abs/2203.02155)
>
> **«impossible»** · «it is impossible that one can train a system that is aligned to everyone’s preferences at once, or where everyone would endorse the tradeoffs.»

Y la conclusión de esa sección. Es imposible entrenar un sistema alineado a las preferencias de todos a la vez, o uno cuyos compromisos todos acepten. Proponen, como camino posible, modelos que se puedan condicionar a las preferencias de distintos grupos.

## Parte 2 · Las fallas que muestra el paper

La figura nueve tiene ejemplos elegidos para mostrar errores simples. Las respuestas no están elegidas.

### Errores simples de InstructGPT 175B

Tres errores. Le preguntan por qué es importante comer medias después de meditar, y responde «There is no clear answer to this question», con teorías. Acepta una premisa falsa; el paper sospecha que hay pocos prompts así en el entrenamiento. Le preguntan qué pasa si disparás una bala de cañón contra un zapallo, y contesta que no hay forma de predecirlo. Se cubre de más, y el paper sospecha por qué: pidieron a los etiquetadores premiar la humildad epistémica, y el reward model lo aprendió. Y con varias restricciones juntas, como diez películas de los años treinta ambientadas en Francia, rinde peor.

| prompt (figura 9 y sección 4.3) | lo que hace | por qué, según el paper |
|---|---|---|
| «…eat socks after meditating?» | acepta la premisa falsa | pocos prompts así en train |
| «…cannonball… at a pumpkin…» | se cubre: «no way to predict» | se premió la humildad |
| «list 10 movies… 1930's… France» | rinde peor | varias restricciones juntas |

### Cómo un RM aprende a premiar cubrirse

Probamos esa sospecha en chico. Cada respuesta tiene tres rasgos: si es correcta, si se cubre con un «depende», y su largo. El noventa por ciento de las preguntas no tiene respuesta clara. Ahí quien etiqueta premia cubrirse; en las obvias, lo castiga. El reward model ve los rasgos pero no sabe si la pregunta era dudosa: una regresión logística sobre la diferencia de rasgos, que es Bradley-Terry. Y le mostramos una pregunta obvia. Aprendió un peso positivo para cubrirse, cero coma nueve siete. En la pregunta obvia, prefiere la respuesta cubierta con probabilidad cero coma siete siete. Un caso de lo que en Alineación, episodio seis, llamamos falla de especificación.

In [8]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep08_cubrirse.py
__file__ = 'snippets/ep08_cubrirse.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import numpy as np
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(0)
# rasgos de una respuesta: [correcta, se cubre ("depende…"), largo]
def respuesta(n):
    return np.stack([rng.random(n) < 0.7, rng.random(n) < 0.5, rng.random(n)], 1).astype(float)

In [9]:
n = 4000
dudosa = rng.random(n) < 0.9               # 90 %: preguntas sin respuesta clara
A, B = respuesta(n), respuesta(n)
def gusto(r, dudosa):                      # lo que premia quien etiqueta
    cubrirse = np.where(dudosa, +1.2, -1.0)    # humildad, salvo si es obvio
    return 2.0 * r[:, 0] + cubrirse * r[:, 1] + 0.3 * r[:, 2]
gana_A = rng.random(n) < 1 / (1 + np.exp(-(gusto(A, dudosa) - gusto(B, dudosa))))

rm = LogisticRegression().fit(A - B, gana_A)    # Bradley-Terry sobre rasgos
w = rm.coef_[0]
print("pesos del RM [correcta, se cubre, largo]:", w.round(2))
directa, cubierta = np.array([1, 0, 0.2]), np.array([1, 1, 0.8])
p = 1 / (1 + np.exp(-w @ (cubierta - directa)))
print(f"pregunta obvia: P(cubierta ≻ directa) = {p:.2f}")

pesos del RM [correcta, se cubre, largo]: [1.89 0.97 0.39]
pregunta obvia: P(cubierta ≻ directa) = 0.77


### Bradley-Terry sobre rasgos

$$
P(A \succ B) = \sigma\big(w^\top(\varphi_A - \varphi_B)\big)
$$
<small>$\varphi$: los 3 rasgos · $w$: sus pesos · $\sigma$: sigmoide</small>

$$
w = (1{,}89;\; 0{,}97;\; 0{,}39)
$$
<small>correcta · se cubre · largo</small>

$$
w^\top(0;\, 1;\, 0{,}6) = 0{,}97 + 0{,}39 \cdot 0{,}6 = 1{,}20
$$
<small>cubierta menos directa, en la pregunta obvia</small>

El reward model del episodio cuatro, con rasgos en vez de un transformer. Fi es el vector de tres rasgos de una respuesta, w los pesos que se aprenden, y sigma la sigmoide, que lleva cualquier número a una probabilidad entre cero y uno. En el dibujo, la curva: cero es empate. Los pesos aprendidos: ser correcta pesa más, pero cubrirse también suma. En la pregunta obvia, las dos respuestas son correctas; la cubierta tiene un uno en cubrirse y es cero coma seis más larga. El producto punto multiplica rasgo por rasgo y suma: uno coma dos. En el dibujo, las tres contribuciones.

> **Ouyang et al., 2022 — sección 5.3** · [arXiv:2203.02155](https://arxiv.org/abs/2203.02155)
>
> **«follow the user's instruction»** · «Perhaps the greatest limitation of our models is that, in most cases, they follow the user’s instruction, even if that could lead to harm in the real world.»

El límite que el paper considera mayor. El modelo, en la mayoría de los casos, sigue la instrucción del usuario aunque pueda causar daño. Es lo que vimos en el episodio seis: con el prompt que pide sesgo, PPO-ptx, el modelo final con proximal policy optimization, es más tóxico que GPT tres.

### Si el RM premia obedecer

Un esquema, con pesos puestos a mano. Cuatro respuestas a un pedido agresivo, con dos rasgos: cuánto cumple y cuánto daño hace. Un reward model que premia sobre todo cumplir, como uno entrenado priorizando ser útil. Otro en el que el daño pesa de verdad. Elegimos la mejor de cuatro según cada uno. El útil elige la que obedece, con daño cero coma nueve. El cauto elige negarse con amabilidad. La política solo optimiza lo que el reward model mide: si el daño casi no pesa, lo ignora.

In [10]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep08_obedece.py
__file__ = 'snippets/ep08_obedece.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import numpy as np
rng = np.random.default_rng(0)

In [11]:
# 4 respuestas candidatas a «escribí algo agresivo sobre X»
#            [cumple el pedido, daño]
candidatas = np.array([[1.0, 0.9],          # obedece
                       [0.6, 0.4],          # obedece a medias
                       [0.2, 0.0],          # se niega con amabilidad
                       [0.0, 0.0]])         # ignora el pedido
rm_util = np.array([2.0, -0.2])             # entrenado priorizando ser útil
rm_cauto = np.array([2.0, -3.0])            # si el daño pesara de verdad
for nombre, w in [("RM útil", rm_util), ("RM cauto", rm_cauto)]:
    puntajes = candidatas @ w
    elegida = puntajes.argmax()             # best-of-4: la que el RM prefiere
    print(f"{nombre:8} puntajes {puntajes.round(1) + 0.0} → elige {elegida}"
          f" (daño {candidatas[elegida, 1]})")

RM útil  puntajes [1.8 1.1 0.4 0. ] → elige 0 (daño 0.9)
RM cauto puntajes [-0.7  0.   0.4  0. ] → elige 2 (daño 0.0)


## Parte 3 · Lo que no resolvió

Sesgo, rechazos y cómo decidir a quién representar. El paper lo deja escrito como pregunta abierta.

### Cómo se mide el sesgo en pares de frases

Winogender y CrowS-Pairs tienen pares de frases que difieren en un detalle, por ejemplo un pronombre. Un modelo sin sesgo no debería preferir ninguna. La entropía en bits: uno si no elige, cero si está segurísimo. La probabilidad de cada frase del par sale de comparar sus log-probabilidades. Y los números del paper. Iguales: entropía uno. Un nat de diferencia: cero coma ocho cuatro. Tres: cero coma dos ocho. En la Tabla catorce, PPO-ptx no mejora a GPT tres. Y cuando la instrucción le pide ser respetuoso, la entropía baja: más seguro, y por esta métrica, más sesgado.

In [12]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep08_entropia.py
__file__ = 'snippets/ep08_entropia.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import numpy as np

In [13]:
def entropia(p):                           # en bits: 1 = no elige, 0 = segurísimo
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

# log-probabilidades (inventadas) de las dos frases de un par
for lp1, lp2 in [(-20.0, -20.0), (-20.0, -21.0), (-20.0, -23.0)]:
    p1 = np.exp(lp1) / (np.exp(lp1) + np.exp(lp2))   # P(frase 1 | par)
    print(f"log p: {lp1:.0f} y {lp2:.0f} → p = {p1:.2f}, entropía {entropia(p1):.2f}")

# Tabla 14, CrowS-Pairs, 175B: entropía (más alta = menos sesgo)
print("GPT básico 0.410 | PPO-ptx básico 0.413 | PPO-ptx respetuoso 0.243")

log p: -20 y -20 → p = 0.50, entropía 1.00
log p: -20 y -21 → p = 0.73, entropía 0.84
log p: -20 y -23 → p = 0.95, entropía 0.28
GPT básico 0.410 | PPO-ptx básico 0.413 | PPO-ptx respetuoso 0.243


### Entropía de un par

$$
p = \frac{e^{\ell_1}}{e^{\ell_1} + e^{\ell_2}} = \sigma(\ell_1 - \ell_2)
$$
<small>$\ell_i$: log-probabilidad de la frase $i$</small>

$$
H(p) = -p\log_2 p - (1-p)\log_2(1-p)
$$
<small>1 bit: no elige · 0: elige siempre</small>

$$
\text{CrowS 175B: } 0{,}410 \to 0{,}413 \to 0{,}243
$$
<small>GPT · PPO-ptx · PPO-ptx «respetuoso»</small>

En fórmulas. l uno y l dos son las log-probabilidades de las dos frases. La probabilidad de la primera, dentro del par, es e a la l uno sobre la suma: dividiendo arriba y abajo por e a la l uno, queda la sigmoide de la diferencia. En el dibujo, los tres pares. H es la entropía en bits: log en base dos. Vale uno en p igual a un medio, y cae a cero en los extremos. Los valores del paper en CrowS-Pairs: casi iguales entre GPT tres y PPO-ptx, y más bajos cuando se le pide ser respetuoso. El paper admite que el patrón no es claro.

### Preguntas abiertas (sección 5.4)

- Negarse a pedidos dañinos: «subsequent iterations»
- Datos adversariales: buscar el peor caso
- Otra señal: editar o criticar en vez de comparar
- Un proceso transparente que represente a los afectados

La lista de preguntas abiertas. Que el modelo se niegue a ciertas instrucciones: dicen que sus técnicas sirven y que lo van a explorar en próximas iteraciones. Datos adversariales: que los etiquetadores busquen el peor comportamiento y se agregue al dataset. Comparar dos respuestas no tiene por qué ser la mejor señal: se podría editar la respuesta o escribir una crítica. Esa idea vuelve en el episodio diez. Y la más grande, según ellos: cómo diseñar un proceso de alineación transparente, que represente a la gente afectada.

## Resumiendo

- Se alinea a ~40 etiquetadores, sus instrucciones y los clientes
- Acuerdo 73 % entre personas; techo de un RM ≈ 84 % (nuestro cálculo)
- Una etiqueta por par y un promedio esconden el desacuerdo
- Fallas: premisas falsas, cubrirse de más, obedecer lo dañino
- Sesgo sin mejora; «respetuoso» lo empeora por entropía
- Próximo: el pipeline entero, en chico